## Day 6 - LSTM Model

##  Prepare Sequence Data

In [6]:
!pip install torch torchvision torchaudio

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load clean data
daily = pd.read_csv('../data/prophet_ready.csv', parse_dates=['ds'])
values = daily['y'].values.reshape(-1, 1)

# Scale to [0, 1] — LSTM trains better on normalized data
scaler = MinMaxScaler()
values_scaled = scaler.fit_transform(values)

# Create sequences: 30 days in → predict 1 day out
def create_sequences(data, lookback=30):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])   # input: 30 days
        y.append(data[i+lookback])     # output: next day
    return np.array(X), np.array(y)

lookback = 30
X, y = create_sequences(values_scaled, lookback)
print(f"X shape: {X.shape}")  # (samples, 30, 1)
print(f"y shape: {y.shape}")  # (samples, 1)

X shape: (709, 30, 1)
y shape: (709, 1)


## Train-Test Split

In [9]:
split = int(len(X) * 0.8)   # 80% train, 20% test

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test)

# DataLoader batches data for training
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=False)

## Define the LSTM Model

In [10]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # x shape: (batch, sequence, features)
        lstm_out, _ = self.lstm(x)
        # Take last time step's output
        out = self.fc(lstm_out[:, -1, :])
        return out

model = LSTMModel()
print(model)

LSTMModel(
  (lstm): LSTM(1, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


## Train the Model

In [11]:
criterion = nn.MSELoss()                          # Mean Squared Error loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 50
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()          # Clear previous gradients
        predictions = model(X_batch)   # Forward pass
        loss = criterion(predictions, y_batch)  # Calculate loss
        loss.backward()               # Backpropagation
        optimizer.step()              # Update weights
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.6f}")

print("Training complete!")

Epoch 10/50 — Loss: 0.008612
Epoch 20/50 — Loss: 0.007847
Epoch 30/50 — Loss: 0.005729
Epoch 40/50 — Loss: 0.004358
Epoch 50/50 — Loss: 0.004104
Training complete!


## Evaluate the LSTM

In [12]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t).numpy()

# Reverse the scaling to get actual revenue values
y_pred = scaler.inverse_transform(y_pred_scaled)
y_actual = scaler.inverse_transform(y_test)

# Calculate MAPE
def mape(actual, predicted):
    mask = actual.flatten() != 0
    return np.mean(np.abs((actual.flatten()[mask] - predicted.flatten()[mask]) 
                          / actual.flatten()[mask])) * 100

lstm_mape = mape(y_actual, y_pred)
print(f"LSTM MAPE: {lstm_mape:.2f}%")

LSTM MAPE: 38.16%


##  Log LSTM to MLflow

In [15]:
# import mlflow

# with mlflow.start_run(run_name="LSTM_Baseline"):
#     mlflow.log_param("model_type", "LSTM")
#     mlflow.log_param("hidden_size", 64)
#     mlflow.log_param("num_layers", 2)
#     mlflow.log_param("lookback", lookback)
#     mlflow.log_param("epochs", EPOCHS)
#     mlflow.log_metric("MAPE", lstm_mape)
    
    # # Save model
    # torch.save(model.state_dict(), 'models/lstm_baseline.pth')
    # mlflow.log_artifact('models/lstm_baseline.pth')
    # print("LSTM logged to MLflow!")

    # Save model locally

import os

os.makedirs("../models", exist_ok=True)

torch.save(
    model.state_dict(),
    "../models/lstm_baseline.pth"
)

print("Model saved successfully!")
print(f"LSTM MAPE = {lstm_mape:.2f}%")

Model saved successfully!
LSTM MAPE = 38.16%
